In [ ]:
import sys
sys.path.insert(0, '/home/frlab/mj_opt/mj_sim/quadruped')

import numpy as np
import mujoco
import mujoco.viewer
import pinocchio as pin
from scipy.spatial import ConvexHull
from core    import FloatingBaseRobotState, Pinocchio_Wrapper, Mujoco_Kernel, LEG_KEYS, LEG_IDX
from control import GaitScheduler, LegController, SRBDModel
from utils   import SimScheduler, Visualizer

%load_ext autoreload
%autoreload 2

모델 경로

In [ ]:
URDF = '/home/frlab/mj_opt/models/go2/urdf/go2_description.urdf'
PKG  = '/home/frlab/mj_opt/models/go2'
MJCF = '/home/frlab/mj_opt/models/scenes/go2/scene.xml'

데코레이터 종류, @staticmethod (정적 메서드), 이렇게 함수를 구성하면 self 인자 없이 method 방법처럼 class명.함수명으로 사용 가능

@dataclass (데이터 클래스), 클래스가 데이터를 저장할 때 파이썬이 표준 메서드 __init__이나 __repr__를 생성해줌

@property (getter), class의 데이터를 사용하고 싶을 때 안에서 함수로 계산해놓고 변수처럼 사용 class명.total_mass

@classmethod, class 자체를 인자로 받아서 여러 객체를 생성할 때 사용, urdf, mjcf를 받아서 원하는 데이터들 생성같은 역할

상태 초기화, Orchestrator 호출, 제어기 호출

In [ ]:
state    = FloatingBaseRobotState()
wrapper  = Pinocchio_Wrapper(URDF, PKG)
joint_names = [wrapper.model.names[i] for i in range(2, wrapper.model.njoints)]
kernel   = Mujoco_Kernel(MJCF, joint_names_pin_order=joint_names)
kernel.register_foot_geoms({"FL": "FL", "FR": "FR", "RL": "RL", "RR": "RR"})

gait     = GaitScheduler(gait_hz=1.0, duty=0.8)
leg_ctrl = LegController(wrapper, gait, kernel)
srbd     = SRBDModel(wrapper, list(LEG_KEYS))

print("✅ 시스템 초기화 완료")

print(f"🚀 [확인용] 로봇의 총 질량은: {wrapper.m} kg 입니다.")

Main Loop (상태 갱신, 제어, 뷰어, 시각화)

In [ ]:
_printed = False 

def on_control(t):
    global _printed  
    
    kernel.update_robot_state(state)
    wrapper.update_model(state.q, state.dq)
    
    # SRBD 모델 빌드 + 출력 (t>0.5에서 한 번만)
    if t > 0.5 and not _printed:
        srbd.update_continuous_dynamics()
        srbd.update_discrete_dynamics(dt=0.01)   # 100Hz MPC 가정
        srbd.print_matrices()
        _printed = False
        
    tau = leg_ctrl.compute(t)
    kernel.ctrl_tau = tau

_dbg_count = 0

def on_render(t, visualizer):
    global _dbg_count

    # 1. 접촉 데이터 가져오기
    _, points = kernel.get_foot_contact_state()
    
    # [1] 3발 이상 접촉: 지지 다각형 (Support Polygon) - 초록색
    if len(points) >= 3:
        pts = np.array(points)
        try:
            hull = ConvexHull(pts[:, :2])
            z = pts[:, 2].mean()
            verts = pts[hull.vertices]
            num_verts = len(verts)
            for j in range(num_verts):
                p1 = np.array([verts[j][0], verts[j][1], z])
                p2 = np.array([verts[(j+1) % num_verts][0],
                               verts[(j+1) % num_verts][1], z])
                visualizer.draw_line(p1, p2, rgba=(0, 1, 0, 1), thickness=0.005)
        except Exception:
            pass

    # [2] 딱 2발만 접촉 (Trot 보행 중): 지지선 (Support Line) - 파란색
    elif len(points) == 2:
        p1 = np.array(points[0])
        p2 = np.array(points[1])
        
        # 선이 땅에 묻히지 않게 두 점의 평균 높이 사용
        z_height = (p1[2] + p2[2]) / 2.0
        p1[2] = z_height
        p2[2] = z_height
        
        visualizer.draw_line(p1, p2, rgba=(0, 0, 1, 1), thickness=0.005)

In [ ]:
kernel.reset_to_keyframe()
sched = SimScheduler(kernel.model, kernel.data, ctrl_hz=500, render_hz=60)
sched.run(on_control=on_control, on_render=on_render)